In [11]:
# Essential packages
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [12]:
file_name = 'Asm1_dataset26.csv'
df = pd.read_csv(file_name)
df.head(5)

,Hectares,Agriblock,Variety,Soil Types,Seedrate(in Kg),LP_Mainfield(in Tonnes),Nursery,Nursery area (Cents),LP_nurseryarea(in Tonnes),DAP_20days,...,Wind Direction_D1_D30,Wind Direction_D31_D60,Wind Direction_D61_D90,Wind Direction_D91_D120,Relative Humidity_D1_D30,Relative Humidity_D31_D60,Relative Humidity_D61_D90,Relative Humidity_D91_D120,Trash(in bundles),Paddy yield(in Kg)
0,6,Cuddalore,CO_43,alluvial,150,75.0,dry,120,6,240,...,SW,W,NNW,WSW,72.0,78,88,85,540,35028
1,6,Kurinjipadi,ponmani,clay,150,75.0,wet,120,6,240,...,NW,S,SE,SSE,64.6,85,84,87,600,35412
2,6,Panruti,delux ponni,alluvial,150,75.0,dry,120,6,240,...,ENE,NE,NNE,W,85.0,96,84,79,600,36300
3,6,Kallakurichi,CO_43,clay,150,75.0,wet,120,6,240,...,--,WNW,SE,S,88.5,95,81,84,540,35016
4,6,Sankarapuram,ponmani,alluvial,150,75.0,dry,120,6,240,...,SSE,W,SW,NW,72.7,91,83,81,600,34044


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2789 entries, 0 to 2788
Data columns (total 45 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Hectares                            2789 non-null   int64  
 1   Agriblock                           2789 non-null   object 
 2   Variety                             2789 non-null   object 
 3   Soil Types                          2789 non-null   object 
 4   Seedrate(in Kg)                     2789 non-null   int64  
 5   LP_Mainfield(in Tonnes)             2789 non-null   float64
 6   Nursery                             2789 non-null   object 
 7   Nursery area (Cents)                2789 non-null   int64  
 8   LP_nurseryarea(in Tonnes)           2789 non-null   int64  
 9   DAP_20days                          2789 non-null   int64  
 10  Weed28D_thiobencarb                 2789 non-null   int64  
 11  Urea_40Days                         2789 no

## Task 1. Data Preparation
* What variables are included in the modelling, their roles and measurement level set and why?
* Any data issues addressed? Describe the step to transform/clean the data
* Create a new binary column named 'isAboveAvg' as the target variable, where the value is set to 1 if the paddy field per hectare exceeds the mean paddy yield per hectare, and 0 otherwise. Reorganize the dataset by removing potentially noisy or non-informative columns, such as Hectares, Paddy Yield, etc.
* Identify and remove highly correlated columns with a pairwise correlation coefficient greater than 0.98

In [14]:
# Cleaning the data
df.columns = df.columns.str.strip() # Column Hectares has a redundant space
print(f'Original dataset: {df.shape}')

# Identify duplicated rows
print(f'Number of duplicated rows: {df.duplicated().sum()}')

# Remove duplicated rows
df = df.drop_duplicates(keep='first')
print(f'Cleaned dataset: {df.shape}')


Original dataset: (2789, 45)
Number of duplicated rows: 161
Cleaned dataset: (2628, 45)


In [15]:
# Identify missing values
missing_value = df.isnull().sum()
print(f'Missing values per columns:\n{missing_value[missing_value > 0]}')

Missing values per columns:
Min temp_D1_D30      107
Min temp_D31_D60     107
Min temp_D61_D90     107
Min temp_D91_D120    107
dtype: int64


In [16]:
# Impute missing values with its median
cols = ['Min temp_D1_D30', 'Min temp_D31_D60', 'Min temp_D61_D90', 'Min temp_D91_D120']
for col in cols:
    df[col] = df[col].fillna(df[col].median())
    print(col, df[col].unique().tolist())

# Replace '--' with NA in categorical variables
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col] = df[col].replace('--', 'NA')
    print(col, df[col].unique().tolist())

Min temp_D1_D30 [18.5, 19.5, 20.0, 19.0, 20.5, 18.0]
Min temp_D31_D60 [16.0, 18.5, 18.0, 17.0, 17.5, 15.5]
Min temp_D61_D90 [15.5, 17.0, 17.5, 16.5, 18.0, 15.0]
Min temp_D91_D120 [16.0, 18.0, 15.5, 16.5, 15.0]
Agriblock ['Cuddalore', 'Kurinjipadi', 'Panruti', 'Kallakurichi', 'Sankarapuram', 'Chinnasalem']
Variety ['CO_43', 'ponmani', 'delux ponni']
Soil Types ['alluvial', 'clay']
Nursery ['dry', 'wet']
Wind Direction_D1_D30 ['SW', 'NW', 'ENE', 'NA', 'SSE', 'E', 'W']
Wind Direction_D31_D60 ['W', 'S', 'NE', 'WNW', 'ENE', 'NA']
Wind Direction_D61_D90 ['NNW', 'SE', 'NNE', 'SW', 'NE', 'NA']
Wind Direction_D91_D120 ['WSW', 'SSE', 'W', 'S', 'NW', 'NNW', 'NA']


In [17]:
# Calculate the mean of the paddy yield per hectares
df['yield_per_hec'] = df['Paddy yield(in Kg)']/df['Hectares'] 
mean_yield = df['yield_per_hec'].mean()
print(f'Mean yield per hec: {mean_yield:.2f}')

# Create a column name 'isAboveAvg'
df['isAboveAvg'] = np.where(df['yield_per_hec'] > mean_yield, 1, 0)
df['isAboveAvg'].value_counts()

Mean yield per hec: 5990.15


isAboveAvg
0    1327
1    1301
Name: count, dtype: int64

In [18]:
# Drop noisy columns
drop_cols = ['Hectares', 'Paddy yield(in Kg)', 'yield_per_hec', 'Trash(in bundles)']
df_clean = df.drop(columns = drop_cols)
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2628 entries, 0 to 2788
Data columns (total 43 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Agriblock                           2628 non-null   object 
 1   Variety                             2628 non-null   object 
 2   Soil Types                          2628 non-null   object 
 3   Seedrate(in Kg)                     2628 non-null   int64  
 4   LP_Mainfield(in Tonnes)             2628 non-null   float64
 5   Nursery                             2628 non-null   object 
 6   Nursery area (Cents)                2628 non-null   int64  
 7   LP_nurseryarea(in Tonnes)           2628 non-null   int64  
 8   DAP_20days                          2628 non-null   int64  
 9   Weed28D_thiobencarb                 2628 non-null   int64  
 10  Urea_40Days                         2628 non-null   float64
 11  Potassh_50Days                      2628 non-nul

### Variables included in modelling

| Variable group | Role | Measurement level | Justification |
|---|---|---|---|
| `isAboveAvg` | Target variable | Binary categorical | This indicates whether paddy yield per hectare is above the dataset mean. It is created from `yield_per_hec` and used as the class label. |
| `Agriblock`, `Variety`, `Soil Types`, `Nursery`, and wind direction variables | Input variables | Nominal categorical | These variables describe location, crop variety, soil type, nursery method, and wind direction. They are retained because they may influence yield performance and are converted with one-hot encoding before modelling. |
| Seed rate, fertiliser, pesticide, rainfall, irrigation, temperature, wind speed, and relative humidity variables | Input variables | Numeric continuous or discrete | These variables describe crop management and environmental conditions that may affect whether the field exceeds the average yield per hectare. |
| `Hectares`, `Paddy yield(in Kg)`, and `yield_per_hec` | Removed variables | Numeric | These variables are used to construct the target variable or directly reveal the yield outcome, so keeping them would cause target leakage. |
| `Trash(in bundles)` | Removed variable | Numeric | This variable is treated as noisy or less informative for predicting whether paddy yield per hectare is above average. |
| Variables with pairwise correlation coefficient greater than 0.98 | Removed variables | Numeric or encoded binary | Highly correlated variables provide redundant information, so one variable from each highly correlated pair is removed to reduce multicollinearity. |


In [19]:
# Correlation Analysis
## Encode categorical columns with One-hot encoding
df_encoded = pd.get_dummies(df_clean, columns=cat_cols, drop_first=True)

# Separate input and target variables
X = df_encoded.drop(columns=['isAboveAvg'])
y = df_encoded['isAboveAvg']

corr_matrix = X.corr().abs()

# Find highly correlated pair-wise columns
upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_cols = [col for col in upper_triangle.columns if any(upper_triangle[col] > 0.98)]

# Drop highly correlated columns
df_final = X.drop(columns=high_corr_cols)

print(f'There are {len(high_corr_cols)} highly correlated columns')
print(f'Highly correlated columns to drop:\n{chr(10).join(high_corr_cols)}')

There are 15 highly correlated columns
Highly correlated columns to drop:
LP_Mainfield(in Tonnes)
Nursery area (Cents)
LP_nurseryarea(in Tonnes)
DAP_20days
Weed28D_thiobencarb
Urea_40Days
Potassh_50Days
Micronutrients_70Days
Pest_60Day(in ml)
30DAI(in mm)
30_50DAI(in mm)
51_70AI(in mm)
71_105DRain(in mm)
71_105DAI(in mm)
Min temp_D61_D90
